In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# ===========================================================================
# CÉLULA 1 - Configurações
# ===========================================================================
CATALOG        = "workspace"
SCHEMA_LANDING = "landing"
SCHEMA_BRONZE  = "bronze"
VOLUME_PATH    = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/dados"

TABELAS = [
    "clientes",
    "categorias",
    "produtos",
    "pedidos",
    "itens_pedido"
]


In [0]:
#===========================================================================
# CÉLULA 2 - Criar schema BRONZE
# ===========================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
print(f"Schema '{SCHEMA_BRONZE}' pronto.")


In [0]:
# ===========================================================================
# CÉLULA 3 - Loop: CSV (Landing) → Delta Lake (Bronze)
# ===========================================================================
resultados = {}

for tabela in TABELAS:
    print(f"\n>>> Processando: {tabela}")
    try:
        # Lê o CSV gerado pelo notebook 00_landing
        caminho_csv = f"{VOLUME_PATH}/{tabela}"
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(caminho_csv)

        qtd = df.count()
        print(f"    Registros lidos do CSV: {qtd}")

        # Adiciona colunas de controle (audit columns)
        df_bronze = df \
            .withColumn("_bronze_ingestion_ts", current_timestamp()) \
            .withColumn("_source_table", lit(tabela)) \
            .withColumn("_source_format", lit("csv"))

        # Grava em Delta Lake (modo overwrite - carga full)
        tabela_destino = f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}"
        df_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(tabela_destino)

        print(f"    Salvo como Delta Lake: {tabela_destino}")
        resultados[tabela] = {"status": "OK", "registros": qtd}

    except Exception as e:
        print(f"    ERRO: {e}")
        resultados[tabela] = {"status": "ERRO", "mensagem": str(e)}


In [0]:
#===========================================================================
# CÉLULA 4 - Relatório
# ===========================================================================
print("\n" + "="*50)
print("RELATÓRIO DE INGESTÃO - BRONZE")
print("="*50)
for tabela, info in resultados.items():
    status = info.get("status")
    regs   = info.get("registros", "-")
    print(f"  {tabela:<20} | Status: {status:<5} | Registros: {regs}")

In [0]:
#===========================================================================
# CÉLULA 5 - Verificação: listar tabelas no schema Bronze
# ===========================================================================
print("\nTabelas no schema BRONZE:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_BRONZE}"))

In [0]:
#===========================================================================
# CÉLULA 6 - Amostra dos dados
# ===========================================================================
print("\nAmostra - clientes (Bronze):")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA_BRONZE}.clientes LIMIT 5"))

print("\nAmostra - pedidos (Bronze):")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA_BRONZE}.pedidos LIMIT 5"))
